# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [2]:
import math
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange


In [26]:
def process_input_dataframe(dataframe):
    # 1. Convert data types
    # 2. Remove lines with empty values
    # 3. Shuffle to introduce randomness
    # 4. Normalize values
    dataframe.apply(pd.to_numeric, errors='coerce')

    dataframe.dropna(inplace=True)

    return dataframe.sample(frac=1)


def normalize_array(array, min_value, max_value):
    return (array - min_value) / (max_value - min_value)


def calculate_normalization_boundaries_for_columns(dataframe, target_columns):
    boundaries = {}

    for column_name in target_columns:
        boundaries[column_name] = {
            'min': get_column_min(dataframe, column_name),
            'max': get_column_max(dataframe, column_name)
        }

    return boundaries


def normalize_dataframe(dataframe, normalization_boundaries_dict):
    for column_name, boundaries_dict in normalization_boundaries_dict.items():
        min_value = boundaries_dict['min']
        max_value = boundaries_dict['max']

        column_arr = np.array(dataframe[column_name])

        dataframe[column_name] = normalize_array(column_arr, min_value, max_value)


def get_column_min(dataframe, column_name):
    return dataframe[column_name].min()


def get_column_max(dataframe, column_name):
    return dataframe[column_name].max()

#### Reading and pre-processing data

In [117]:
train_file_path = 'data/biogas/biogas_training_with_type.csv'
test_file_name_path = 'data/biogas/biogas_testing_with_type.csv'


data_columns = ['Input', 'Humidity', 'Temperature', 'Holding Time']
result_columns = ['Output']
all_columns = data_columns + result_columns

train_df = pd.read_csv(train_file_path, header=None, names=all_columns)
test_df = pd.read_csv(test_file_name_path, header=None, names=all_columns)

train_df = process_input_dataframe(train_df)
test_df = process_input_dataframe(test_df)

#### Displaying basic information about the initial data


In [105]:
def display_dataframe_info(dataframe):
    print('****** General Information ******')
    display(dataframe.info())
    display(dataframe.describe())

    print('****** Data Sample ******')
    display(dataframe.head())


In [106]:
print('==== Train Data Info ====')
display_dataframe_info(train_df)
print('========================', end='\n\n')

print('==== Test Data Info ====')
display_dataframe_info(test_df)
print('=========================')

==== Train Data Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 77 to 29
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Input         200 non-null    int64  
 1   Humidity      200 non-null    float64
 2   Temperature   200 non-null    float64
 3   Holding Time  200 non-null    int64  
 4   Output        200 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 9.4 KB


None

,Input,Humidity,Temperature,Holding Time,Output
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,277.285000,76.160253,41.601574,25.200000,2.375000
std,128.118208,9.166107,7.032012,8.674459,0.759777
min,50.000000,60.325130,30.501780,10.000000,1.000000
25%,168.500000,68.740083,35.128928,17.000000,2.000000
50%,291.500000,76.790133,41.081841,26.000000,3.000000
75%,386.000000,84.384748,47.403338,32.000000,3.000000
max,499.000000,89.991530,54.749006,39.000000,3.000000


****** Data Sample ******


,Input,Humidity,Temperature,Holding Time,Output
77,404,62.505022,42.461055,31,2
89,169,79.476309,54.662881,30,2
171,433,80.151021,42.568010,31,2
166,65,73.516324,40.978373,23,1
55,107,86.631125,48.686773,17,1



==== Test Data Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 495 to 152
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Input         50 non-null     float64
 1   Humidity      50 non-null     float64
 2   Temperature   50 non-null     int64  
 3   Holding Time  50 non-null     float64
 4   Output        50 non-null     int64  
dtypes: float64(3), int64(2)
memory usage: 2.3 KB


None

,Input,Humidity,Temperature,Holding Time,Output
count,50.000000,50.000000,50.000000,50.000000,50.000000
mean,75.043101,41.833958,24.820000,7.483660,2.500000
std,8.556196,6.727222,9.475403,0.591379,0.646813
min,60.397949,30.013009,10.000000,6.518394,1.000000
25%,68.411143,36.502108,15.250000,6.958562,2.000000
50%,75.907896,41.324816,26.000000,7.590975,3.000000
75%,81.698685,46.702788,32.750000,7.934402,3.000000
max,88.957659,54.292802,39.000000,8.373460,3.000000


****** Data Sample ******


,Input,Humidity,Temperature,Holding Time,Output
495,71.846446,33.052199,11,8.274173,3
369,87.789026,37.619531,32,6.518394,3
463,71.854507,44.081889,12,7.182133,3
363,85.336015,44.971637,26,7.610402,1
493,77.113319,43.496027,33,7.015883,2


#### Determining normalization boundaries and performing normalization fot both the training and testing sets


In [118]:
normalization_boundaries_dict = calculate_normalization_boundaries_for_columns(train_df, [])

print('====== Data Normalization ======', end='\n')

print('Based on the input training data, each column will be normalized using the following boundaries:', end='\n')

for column_name, boundaries_dict in normalization_boundaries_dict.items():
    min_value = boundaries_dict['min']
    max_value = boundaries_dict['max']

    print(f'{column_name}: [{min_value}; {max_value}]', end='\n')

normalize_dataframe(train_df, normalization_boundaries_dict)
normalize_dataframe(test_df, normalization_boundaries_dict)

print('\nData has been normalized')
print('================================', end='\n')



====== Data Normalization ======
Based on the input training data, each column will be normalized using the following boundaries:

Data has been normalized


In [119]:
print('==== Train Data Normalized Info ====')
display_dataframe_info(train_df)
print('========================', end='\n\n')

print('==== Test Data Normalized Info ====')
display_dataframe_info(test_df)
print('=========================')

==== Train Data Normalized Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 186 to 70
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Input         200 non-null    int64  
 1   Humidity      200 non-null    float64
 2   Temperature   200 non-null    float64
 3   Holding Time  200 non-null    int64  
 4   Output        200 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 9.4 KB


None

,Input,Humidity,Temperature,Holding Time,Output
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,277.285000,76.160253,41.601574,25.200000,2.375000
std,128.118208,9.166107,7.032012,8.674459,0.759777
min,50.000000,60.325130,30.501780,10.000000,1.000000
25%,168.500000,68.740083,35.128928,17.000000,2.000000
50%,291.500000,76.790133,41.081841,26.000000,3.000000
75%,386.000000,84.384748,47.403338,32.000000,3.000000
max,499.000000,89.991530,54.749006,39.000000,3.000000


****** Data Sample ******


,Input,Humidity,Temperature,Holding Time,Output
186,372,82.654118,52.347315,19,2
181,494,76.828139,38.372640,38,2
68,157,62.545131,30.501780,20,3
123,398,70.600567,45.265501,10,1
39,192,88.501859,49.985260,39,2



==== Test Data Normalized Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 343 to 393
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Input         50 non-null     float64
 1   Humidity      50 non-null     float64
 2   Temperature   50 non-null     int64  
 3   Holding Time  50 non-null     float64
 4   Output        50 non-null     int64  
dtypes: float64(3), int64(2)
memory usage: 2.3 KB


None

,Input,Humidity,Temperature,Holding Time,Output
count,50.000000,50.000000,50.000000,50.000000,50.000000
mean,75.043101,41.833958,24.820000,7.483660,2.500000
std,8.556196,6.727222,9.475403,0.591379,0.646813
min,60.397949,30.013009,10.000000,6.518394,1.000000
25%,68.411143,36.502108,15.250000,6.958562,2.000000
50%,75.907896,41.324816,26.000000,7.590975,3.000000
75%,81.698685,46.702788,32.750000,7.934402,3.000000
max,88.957659,54.292802,39.000000,8.373460,3.000000


****** Data Sample ******


,Input,Humidity,Temperature,Holding Time,Output
343,87.799766,47.387902,26,6.726947,3
98,78.211027,32.637356,32,8.300836,3
241,69.796223,45.110434,26,8.254679,3
219,68.888205,35.461011,10,7.178060,1
71,82.419603,47.369623,11,7.559301,3


#### Declaring Utils for Layers


In [90]:
def kronecker_delta(i, j):
    return 1 if i == j else 0

#### Declaring Layers


Fuzzifying Layer

In [91]:
class FuzzifyingLayer:
    def __init__(self, membership_functions_count, input_vector_length, c_initial, s_initial):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length
        self.c = []
        self.s = []

        for i in range(self.input_vector_length):
            self.c.append(c_initial)
            self.s.append(s_initial)

    def fuzzify(self, input_vector):
        memberships_matrix = []

        for i in range(self.input_vector_length):
            x = input_vector[i]
            x_memberships = []

            for j in range(self.membership_functions_count):
                membership = self.calculate_membership_for_x(x, i, j)
                x_memberships.append(membership)

            memberships_matrix.append(x_memberships)

        # vector_memberships = [
        #     x1: [ mu1(x1), mu2(x1), ..., mu(x1)]
        #     x1: [ mu1(x1), mu2(x1), ..., mu(x1)]
        #     ...
        #     x: [ mu1(x), mu2(x), ..., mu(x)]
        # ]

        # print(f"memberships_matrix: {memberships_matrix}", end='\n\n')
        # print(f"s: {self.c}", end='\n\n')
        # print(f"c: {self.s}", end='\n\n')

        return memberships_matrix

    def calculate_membership_for_x(self, x, i, j):
        c, s = self.c[i][j], self.s[i][j]

        membership = 1 / (1 + (((x - c) / s) ** 2))

        return membership

    def train(self, v_c, v_s, predicted_value, actual_values, weights, input_vector):
        c_copy = np.empty_like(self.c)
        s_copy = np.empty_like(self.s)

        for r in range(len(self.c)):
            for l in range(len(self.c[r])):
                c_copy[r][l] = self.calculate_new_c(r, l, weights, v_c, predicted_value, actual_values, input_vector)
                s_copy[r][l] = self.calculate_new_s(r, l, weights, v_s, predicted_value, actual_values, input_vector)

        self.c = c_copy
        self.s = s_copy

    def calculate_new_c(self, r, j, weights, v_c, predicted_value, actual_value, input_vector):
        memberships_matrix = self.fuzzify(input_vector)

        return self.c[r][j] - v_c * self.calculate_de_to_dc(r, j, weights, predicted_value, actual_value, input_vector,
                                                            memberships_matrix)

    def calculate_new_s(self, r, j, weights, v_s, predicted_value, actual_value, input_vector):
        memberships_matrix = self.fuzzify(input_vector)

        return self.s[r][j] - v_s * self.calculate_de_to_ds(r, j, weights, predicted_value, actual_value,
                                                            input_vector, memberships_matrix)

    def calculate_de_to_dc(self, r, j, weights, predicted_value, actual_value, input_vector, memberships_matrix):
        error_derivative = np.sum(predicted_value - actual_value)

        weighted_outputs = 0

        for l in range(len(weights)):
            weighted_outputs += weights[l] * self.calculate_dy_to_dc(l, r, j, input_vector, memberships_matrix)

        return error_derivative * weighted_outputs

    def calculate_de_to_ds(self, r, j, weights, predicted_value, actual_value, input_vector, memberships_matrix):
        error_derivative = np.sum(predicted_value - actual_value)

        weighted_outputs = 0

        for l in range(len(weights)):
            weighted_outputs += weights[l] * self.calculate_dy_to_ds(l, r, j, input_vector, memberships_matrix)

        return error_derivative * weighted_outputs

    def calculate_dy_to_dc(self, l, r, j, input_vector, memberships_matrix):
        m = self.calculate_m(memberships_matrix)
        t = self.calculate_t(l, memberships_matrix)

        first_multiplier = (kronecker_delta(l, r) * m - t) / (m ** 2)
        second_multiplier = self.calculate_t_excluding(r, j, memberships_matrix)
        third_multiplier = self.calculate_dm_to_dc(r, j, input_vector)

        dy_to_dc = first_multiplier * second_multiplier * third_multiplier

        return dy_to_dc

    def calculate_dy_to_ds(self, l, r, j, input_vector, memberships_matrix):
        m = self.calculate_m(memberships_matrix)
        t = self.calculate_t(l, memberships_matrix)

        first_multiplier = (kronecker_delta(l, r) * m - t) / (m ** 2)
        second_multiplier = self.calculate_t_excluding(r, j, memberships_matrix)
        third_multiplier = self.calculate_dm_to_ds(r, j, input_vector)

        dy_to_dc = first_multiplier * second_multiplier * third_multiplier

        return dy_to_dc

    def calculate_dm_to_dc(self, r, j, input_vector):
        x = input_vector[r]
        c = self.c[r][j]
        s = self.s[r][j]

        return math.exp(-(((x - c) ** 2) / (2 * (s ** 2)))) * ((x - c) / (s ** 2))

    def calculate_dm_to_ds(self, r, j, input_vector):
        x = input_vector[r]
        c = self.c[r][j]
        s = self.s[r][j]

        return math.exp(-(((x - c) ** 2) / (2 * (s ** 2)))) * (((x - c) ** 2) / (s ** 3))

    def calculate_m(self, memberships_matrix):
        result = 0

        for p in range(self.membership_functions_count):
            result += self.calculate_t(p, memberships_matrix)

        return result

    def calculate_t(self, l, memberships_matrix):
        result = 1

        for i in range(self.input_vector_length):
            result *= memberships_matrix[i][l]

        return result

    def calculate_t_excluding(self, r, j, memberships_matrix):
        result = 1

        for i in range(self.input_vector_length):
            if i != j:
                result *= memberships_matrix[i][r]

        return result

Aggregation Layer


In [92]:
class AggregatingLayer:
    def __init__(self, membership_functions_count, input_vector_length):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length

    def aggregate(self, memberships_matrix):
        aggregated_memberships = []

        for i in range(self.membership_functions_count):
            product = 1

            for j in range(self.input_vector_length):
                product *= memberships_matrix[j][i]

            aggregated_memberships.append(product)

        return aggregated_memberships

Linear Layer

In [93]:
class LinearLayer:
    def __init__(self, membership_functions_count, input_vector_length):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length
        self.weights = np.random.uniform(low=-1, high=1, size=self.membership_functions_count).tolist()

    def process(self, aggregated_memberships):
        weighted_aggregated_memberships_sum = 0
        aggregated_memberships_sum = 0

        for i in range(self.membership_functions_count):
            weighted_aggregated_memberships_sum += self.weights[i] * aggregated_memberships[i]
            aggregated_memberships_sum += aggregated_memberships[i]

        return {
            'f1': weighted_aggregated_memberships_sum,
            'f2': aggregated_memberships_sum,
        }

    def train(self, input_data, expected_output_data, plot_weights=False):
        p_vs = []

        for input_vector in input_data:
            p_v = self.calculate_p_vs(input_vector)
            p_vs.append(p_v)

        pseudo_inverse = np.linalg.pinv(p_vs)
        new_weights = np.matmul(pseudo_inverse, expected_output_data)

        if plot_weights:
            plt.clf()
            plt.plot(self.weights, color='r', label='Previous Weights')
            plt.plot(new_weights, color='g', label='New Weights')
            plt.legend()
            plt.show()

        self.weights = new_weights

    def calculate_p_vs(self, memberships_matrix):
        values = []
        p_vs = []

        for i in range(self.membership_functions_count):
            value = 1

            for j in range(self.input_vector_length):
                value *= memberships_matrix[j][i]

            values.append(value)

        for i in range(len(values)):
            numerator = values[i]
            denominator = 0

            for j in range(len(values)):
                denominator += values[j]

            p_vs.append(numerator / denominator)

        return p_vs

Activation Layer

In [94]:
class ActivationLayer:
    def diffuzify(self, fs_dict):
        # return np.round(fs_dict['f1'] / fs_dict['f2'], 0)
        return fs_dict['f1'] / fs_dict['f2']

### WangMendel Fuzzy Neural Network



In [112]:
class WangMendelFuzzyNeuralNetwork:
    def __init__(self, membership_functions_count, input_vector_length, c_initial, s_initial):
        self.fuzzifying_layer = FuzzifyingLayer(membership_functions_count, input_vector_length, c_initial, s_initial)
        self.aggregating_layer = AggregatingLayer(membership_functions_count, input_vector_length)
        self.linear_layer = LinearLayer(membership_functions_count, input_vector_length)
        self.activation_layer = ActivationLayer()

    def predict_exact(self, input_vector):
        return np.round(self.predict(input_vector), 0)

    def predict(self, input_vector):
        fuzzifying_layer_output = self.fuzzifying_layer.fuzzify(input_vector)
        aggregating_layer_output = self.aggregating_layer.aggregate(fuzzifying_layer_output)
        linear_layer_output = self.linear_layer.process(aggregating_layer_output)
        activation_layer_output = self.activation_layer.diffuzify(linear_layer_output)

        return activation_layer_output

    def predict_for_matrix(self, input_data):
        predictions = []

        for input_vector in input_data:
            prediction = self.predict(input_vector)
            predictions.append(prediction)

        return predictions

    def predict_and_validate(self, input_data, expected_output_data, detailed_output=False):
        predicted_output_data = self.predict_for_matrix(input_data)

        total_predictions = len(predicted_output_data)
        correct_predictions = 0

        for i in range(total_predictions):
            predicted_value = predicted_output_data[i]
            predicted_value_rounded = np.round(predicted_value, 0)
            expected_value = expected_output_data[i]

            is_predicted_equal_to_expected = predicted_value_rounded == expected_value

            if is_predicted_equal_to_expected:
                correct_predictions += 1

            if detailed_output:
                print(
                    f'Predicted: {predicted_value}; Actual: {expected_value}; {'✅' if is_predicted_equal_to_expected else '❌'}')

        mse = (np.square(predicted_output_data - expected_output_data)).mean()
        accuracy = np.round(correct_predictions / total_predictions * 100, 0)

        print(f'MSE: {mse}')
        print(f'Accuracy: {accuracy}')

        return [mse, accuracy]

    def train(self, epochs, training_data_df, c_nu=0.5, s_nu=0.5, max_epochs_with_no_mse_change=2):
        print(f'Started training for {epochs} epochs', end='\n')

        input_data = np.array(training_data_df.iloc[:, :-1].values, dtype=float)
        expected_output_data = np.array(training_data_df.iloc[:, -1].values, dtype=float)

        start_time = time.time()

        current_mse = 1
        epochs_without_mse_change = 0
        completed_epochs = 0

        training_vectors_count = len(input_data)

        for i in trange(epochs, desc='Epochs progress'):
            print(f'Epoch {i + 1} started', end='\n')

            # print(f"====\nCs: {self.fuzzifying_layer.c}\n")
            # print(f"Ss: {self.fuzzifying_layer.s}\n\n====\n")

            self.train_linear_layer(input_data, expected_output_data)

            for j in trange(100, desc='Data passes per epoch progress', leave=False):
                predicted_data = self.predict_for_matrix(input_data)

                for k in range(training_vectors_count):
                    self.fuzzifying_layer.train(
                        c_nu,
                        s_nu,
                        predicted_data,
                        expected_output_data,
                        self.linear_layer.weights,
                        input_data[k]
                    )

            [mse, accuracy] = self.predict_and_validate(input_data, expected_output_data, detailed_output=True)

            previous_mse = current_mse
            current_mse = mse
            mse_change = previous_mse - current_mse

            if mse_change == 0:
                epochs_without_mse_change += 1

            if epochs_without_mse_change == max_epochs_with_no_mse_change:
                print(f'MSE hasn\'t changed during last {max_epochs_with_no_mse_change} epochs; Training ended',
                      end='\n')
                break

            completed_epochs += 1
            print(f'Epoch {i + 1} completed', end='\n')

        end_time = time.time()
        elapsed_time = np.round(end_time - start_time, 0)

        print(f'Finished training in {elapsed_time}s in {completed_epochs} epochs', end='\n')

    def train_linear_layer(self, input_data, expected_output_data):
        input_data_fuzzified = []

        for input_vector in input_data:
            fuzzified_vector = self.fuzzifying_layer.fuzzify(input_vector)
            input_data_fuzzified.append(fuzzified_vector)

        self.linear_layer.train(input_data_fuzzified, expected_output_data)

    def save_to_file(self, file_name):
        c = self.fuzzifying_layer.c
        s = self.fuzzifying_layer.s
        weights = self.linear_layer.weights

        np.savez(f"{file_name}.npz", c=c, s=s, weights=weights)

    def initialize_from_file(self, file_name):
        file = np.load(f"{file_name}.npz")

        self.fuzzifying_layer.c = file['c']
        self.fuzzifying_layer.s = file['s']
        self.linear_layer.weights = file['weights']


#### Training

In [121]:
min_class = get_column_min(train_df, result_columns[0])
max_class = get_column_max(train_df, result_columns[0])

membership_functions_count = 12

# TODO: add utility method to extract x_vector and y
input_vector_length = len(np.array(train_df.iloc[:, :-1].values, dtype=float)[0])

c_initial = np.linspace(0, 400, membership_functions_count).tolist()
s_initial = np.full(membership_functions_count, 0.1).tolist()

wangMendelFuzzyNeuralNetwork = WangMendelFuzzyNeuralNetwork(membership_functions_count, input_vector_length, c_initial,
                                                            s_initial)

wangMendelFuzzyNeuralNetwork.train(10, train_df, 0.5, 0.5, 2)

Started training for 10 epochs


Epochs progress:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


Data passes per epoch progress:   0%|          | 0/100 [00:00<?, ?it/s]

Predicted: 2.351781103950021; Actual: 2.0; ✅
Predicted: 2.3516564327731606; Actual: 2.0; ✅
Predicted: 2.351646829750353; Actual: 3.0; ❌
Predicted: 2.351653434787033; Actual: 1.0; ❌
Predicted: 2.352320900428375; Actual: 2.0; ✅
Predicted: 2.3524750409952007; Actual: 3.0; ❌
Predicted: 2.3516487212407826; Actual: 3.0; ❌
Predicted: 2.351866759584631; Actual: 3.0; ❌
Predicted: 2.3518148543283144; Actual: 1.0; ❌
Predicted: 2.351652736233603; Actual: 2.0; ✅
Predicted: 2.35164996907674; Actual: 3.0; ❌
Predicted: 2.351646150138441; Actual: 3.0; ❌
Predicted: 2.351812498057523; Actual: 3.0; ❌
Predicted: 2.3516462901639295; Actual: 3.0; ❌
Predicted: 2.351658244743562; Actual: 3.0; ❌
Predicted: 2.3516498117167446; Actual: 3.0; ❌
Predicted: 2.3522775529663407; Actual: 3.0; ❌
Predicted: 2.351645825874325; Actual: 1.0; ❌
Predicted: 2.3520154587831557; Actual: 3.0; ❌
Predicted: 2.3517141876296472; Actual: 3.0; ❌
Predicted: 2.351682559441392; Actual: 3.0; ❌
Predicted: 2.351645443957642; Actual: 1.0; ❌
Pr

...

Predicted: 2.4407444044412285; Actual: 2.0; ✅
Predicted: 2.3772093745297136; Actual: 2.0; ✅
Predicted: 2.377593550189842; Actual: 3.0; ❌
Predicted: 2.324235201993442; Actual: 1.0; ❌
Predicted: 2.2721454882929315; Actual: 2.0; ✅
Predicted: 2.3820477914335623; Actual: 3.0; ❌
Predicted: 2.3768475488522256; Actual: 3.0; ❌
Predicted: 2.3691473019665152; Actual: 3.0; ❌
Predicted: 2.3782458866321803; Actual: 1.0; ❌
Predicted: 2.3755971737134622; Actual: 2.0; ✅
Predicted: 2.3770212378222464; Actual: 3.0; ❌
Predicted: 2.3777057052344555; Actual: 3.0; ❌
Predicted: 2.4105888238236304; Actual: 3.0; ❌
Predicted: 2.407118595886149; Actual: 3.0; ❌
Predicted: 2.375163986405885; Actual: 3.0; ❌
Predicted: 2.366449988557208; Actual: 3.0; ❌
Predicted: 2.3884214229526637; Actual: 3.0; ❌
Predicted: 2.378530180514822; Actual: 1.0; ❌
Predicted: 2.2455767389442807; Actual: 3.0; ❌
Predicted: 2.3840055651049865; Actual: 3.0; ❌
Predicted: 2.3785331593816426; Actual: 3.0; ❌
Predicted: 2.378790320172757; Actual: 1.

Data passes per epoch progress:   0%|          | 0/100 [00:00<?, ?it/s]

Predicted: 2.4407444828718834; Actual: 2.0; ✅
Predicted: 2.3772094365251326; Actual: 2.0; ✅
Predicted: 2.37759361927335; Actual: 3.0; ❌
Predicted: 2.3242354746531015; Actual: 1.0; ❌
Predicted: 2.27214504107674; Actual: 2.0; ✅
Predicted: 2.38204734426682; Actual: 3.0; ❌
Predicted: 2.3768476069818294; Actual: 3.0; ❌
Predicted: 2.3691472723249922; Actual: 3.0; ❌
Predicted: 2.3782460224639044; Actual: 1.0; ❌
Predicted: 2.375597207287945; Actual: 2.0; ✅
Predicted: 2.3770212981332963; Actual: 3.0; ❌
Predicted: 2.3777057857913086; Actual: 3.0; ❌
Predicted: 2.410588881967467; Actual: 3.0; ❌
Predicted: 2.407119032021095; Actual: 3.0; ❌
Predicted: 2.375164038589148; Actual: 3.0; ❌
Predicted: 2.366449573623415; Actual: 3.0; ❌
Predicted: 2.388421461867858; Actual: 3.0; ❌
Predicted: 2.378530260756303; Actual: 1.0; ❌
Predicted: 2.2455715289641547; Actual: 3.0; ❌
Predicted: 2.3840056769952485; Actual: 3.0; ❌
Predicted: 2.3785332243736157; Actual: 3.0; ❌
Predicted: 2.3787904040060437; Actual: 1.0; ❌
P

Data passes per epoch progress:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [110]:
wangMendelFuzzyNeuralNetwork.save_to_file('models/regular/biogas/model_37')

#### Testing

In [80]:
# wangMendelFuzzyNeuralNetwork.predict([0.1525, 0.3146, 0.4696, 0.0633])

c_initial = np.linspace(min_class, max_class, membership_functions_count).tolist()
s_initial = np.full(membership_functions_count, 0.2).tolist()

wangMendelFuzzyNeuralNetwork1 = WangMendelFuzzyNeuralNetwork(membership_functions_count,4, c_initial,
                                                             s_initial)
wangMendelFuzzyNeuralNetwork1.initialize_from_file('models/regular/biogas/model_33')

input_data = np.array(test_df.iloc[:, :-1].values, dtype=float)
expected_output_data = np.array(test_df.iloc[:, -1].values, dtype=float)

wangMendelFuzzyNeuralNetwork1.predict_and_validate(input_data, expected_output_data, detailed_output=True)

# wangMendelFuzzyNeuralNetwork.predict([0.1525, 0.3146, 0.4696, 0.0633])

Predicted: 2.4903815782479968; Actual: 2.0; ✅
Predicted: 2.455486109667136; Actual: 3.0; ❌
Predicted: 2.1351030184176603; Actual: 3.0; ❌
Predicted: 2.6518002641461442; Actual: 3.0; ✅
Predicted: 2.3786173799659833; Actual: 2.0; ✅
Predicted: 2.302693641542894; Actual: 3.0; ❌
Predicted: 2.6246466845481726; Actual: 2.0; ❌
Predicted: 2.384072183397112; Actual: 3.0; ❌
Predicted: 2.404095549759154; Actual: 2.0; ✅
Predicted: 2.3254205247335236; Actual: 2.0; ✅
Predicted: 2.4310235512572644; Actual: 2.0; ✅
Predicted: 2.3893061122592694; Actual: 3.0; ❌
Predicted: 2.354024956318377; Actual: 2.0; ✅
Predicted: 2.347452896180027; Actual: 3.0; ❌
Predicted: 2.408325915035274; Actual: 3.0; ❌
Predicted: 2.390381797081924; Actual: 3.0; ❌
Predicted: 2.4292490925483494; Actual: 3.0; ❌
Predicted: 2.3907311871227757; Actual: 1.0; ❌
Predicted: 2.3698690039290753; Actual: 3.0; ❌
Predicted: 2.363718244851912; Actual: 2.0; ✅
Predicted: 2.373184777661735; Actual: 3.0; ❌
Predicted: 2.3695497114747743; Actual: 2.0; 

[0.4102137181828971, 40.0]